# Corrosion inhibitor QSPR: step-by-step walkthrough

Predicting the inhibition efficiency (IE) of organic corrosion inhibitors for carbon steel in 1 M HCl from molecular structure. Run the cells top to bottom; each step reuses the code in `src/`. The full benchmark is `python scripts/run_pipeline.py`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import download_raw, load_raw, build_clean_dataset
from src.features import rdkit_descriptors
from src.models import make_models
from src.evaluate import run_cv

## 2. Get and inspect the data

The Fe-HCl-317 dataset (Pham et al., *RSC Advances*, 2024) is downloaded from its original repository.

In [ ]:
download_raw()
raw = load_raw()
print(raw.shape)
raw.drop(columns='structure').head()

In [ ]:
raw['IE_exp'].describe()

## 3. Standardise structures

Counter-ions are stripped (largest fragment kept), invalid SMILES are dropped, and molecules that become identical after standardisation are merged. Each molecule also gets a Bemis-Murcko scaffold label, used later to build splits that test generalisation to new chemical families.

In [ ]:
df = build_clean_dataset()
print(len(df), 'molecules,', df['scaffold'].nunique(), 'scaffolds')
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(df['IE'], bins=25, color='#2a78d6', edgecolor='white')
ax.set_xlabel('Experimental IE (%)'); ax.set_ylabel('Molecules')
ax.set_title('Distribution of inhibition efficiency')
plt.show()

## 4. Featurise

RDKit computes about 210 two-dimensional descriptors per molecule.

In [ ]:
X = rdkit_descriptors(df['smiles'].tolist())
y = df['IE'].to_numpy()
print(X.shape)
X.iloc[:3, :8]

## 5. Why the split matters

A random split lets close analogues of a test molecule sit in the training set. A scaffold split keeps whole chemical families together, which is closer to screening genuinely new compounds. Expect the scaffold numbers to be worse; that gap is the honest estimate.

In [ ]:
model = make_models(C.SEED)['XGBoost']
groups = df['scaffold'].to_numpy()
for scheme in ['random', 'scaffold']:
    res, oof = run_cv(model, X, y, groups, scheme, n_splits=5, n_repeats=1, seed=C.SEED)
    print(f"{scheme:9s} R2={res['R2']:.2f}  RMSE={res['RMSE']:.1f}  MAE={res['MAE']:.1f}")

## 6. Fit a final model and look inside it with SHAP

In [ ]:
import shap
final = make_models(C.SEED)['XGBoost'].fit(X, y)
Xt = pd.DataFrame(final['filter'].transform(X), columns=final['filter'].get_feature_names_out())
sv = shap.TreeExplainer(final['model']).shap_values(Xt)
shap.summary_plot(sv, Xt, max_display=15)

## 7. Predict new molecules

Needs the saved model from `python scripts/run_pipeline.py`. The similarity column flags molecules far from the training data, where predictions are least reliable.

In [ ]:
from src.predict import predict_ie

predict_ie(['c1ccc2[nH]c(N)nc2c1', 'CCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-]', 'OC(=O)CCN'])